# Welcome to Deep Learning & Fine-Tuning for Bioinformatics! This notebook will guide you step by step — no prior experience needed.

# 🧠 Notebook Guide — Deep Learning & Fine-Tuning for Bioinformatics

## Learning Objectives
- [ ] Build a 1D CNN for DNA/protein sequence classification with PyTorch
- [ ] Implement a proper training loop with learning rate scheduling
- [ ] Apply transfer learning: freeze backbone → fine-tune head → unfreeze layers
- [ ] Implement LoRA (Low-Rank Adaptation) from scratch
- [ ] Build a Protein Transformer with Pre-LN and CLS token
- [ ] Understand ESM2/ProtTrans fine-tuning concepts for interviews

## Why This Matters for HackerRank/Interviews
Bioinformatics ML roles at top companies increasingly require:
- Fine-tuning protein language models (ESM2, ProtTrans, AlphaFold2)
- Efficient adaptation methods (LoRA, adapter layers, prompt tuning)
- Training infrastructure knowledge (distributed training, gradient checkpointing)
- Understanding of when/why to use each approach

---

## 🤖 Claude Integration — Copy These Prompts

**For CNNs on Sequences (first deep learning topic):**
```
Explain 1D CNNs for sequence classification.
How is Conv1d different from Conv2d?
For a DNA sequence of length 200 one-hot encoded to (200, 4):
- What does a Conv1d(4, 32, kernel_size=7) output?
- What does MaxPool1d(2) do to the sequence length?
- Why use Global Average Pooling at the end instead of Flatten?
Walk me through the full architecture: input → Conv1d layers → pool → FC → output.
```

**For the Training Loop:**
```
Explain the PyTorch training loop pattern.
What does each step do:
1. optimizer.zero_grad() — why is this needed?
2. outputs = model(inputs)
3. loss = criterion(outputs, targets)
4. loss.backward() — what does this compute?
5. optimizer.step() — what does this update?
What is gradient clipping and when do I need it?
What is ReduceLROnPlateau and when does it reduce the LR?
```

**For Transfer Learning:**
```
Explain transfer learning and fine-tuning strategy.
When I load a pretrained model:
1. Why freeze the backbone first?
2. When do I unfreeze the backbone?
3. What learning rate should I use for fine-tuning vs training from scratch?
What is "catastrophic forgetting" and how does fine-tuning cause it?
What's the "learning rate warmup" trick and why does it help?
```

**For LoRA (Low-Rank Adaptation — KEY TOPIC):**
```
Explain LoRA (Low-Rank Adaptation) from scratch.
For a weight matrix W of shape (d, d):
- LoRA adds: W' = W + BA where B is (d, r) and A is (r, d), r << d
- Why does this reduce trainable parameters?
- What is the rank r? How do I choose it?
- Why multiply by alpha/r?
- How many parameters does LoRA add vs full fine-tuning?
Show me a LoRALinear class implementation in PyTorch.
What's the difference between LoRA and adapter layers?
```

**For Transformer Architecture:**
```
Explain the Transformer encoder for protein sequences.
What is Pre-Layer Normalization vs Post-Layer Normalization?
Why does Pre-LN improve training stability?
What is the CLS token and how is it used for classification?
For a protein of 500 amino acids:
- Input: token embedding (500, d_model) + positional encoding
- After Transformer encoder: (501, d_model) [+CLS]
- Classification: take CLS token → linear head
Implement a minimal ProteinTransformer class.
```

**For ESM2/ProtTrans fine-tuning:**
```
I want to fine-tune ESM2 for protein function prediction.
Walk me through the complete approach:
1. Load pretrained ESM2 weights
2. Freeze all layers except the final few
3. Add a classification head
4. Fine-tune with small learning rate (1e-4 or less)
5. Gradually unfreeze more layers
What learning rate should I use? (1e-5 to 1e-4 for frozen backbone)
How do I handle proteins longer than the max sequence length (1022 tokens)?
What evaluation metrics should I use?
```

**For AlphaFold2 concepts:**
```
Explain AlphaFold2's key innovations for a technical interview:
1. What is the Evoformer? What does it operate on?
2. What is the MSA (Multiple Sequence Alignment) representation?
3. What is the Pair representation and what does it encode?
4. What is the Invariant Point Attention (IPA)?
5. What does recycling mean in AlphaFold2?
6. What is lDDT-Cα and how does AF2 use it for training?
I don't need a code implementation, just conceptual clarity.
```

---

## Architecture Reference

| Model | Input | Key Layer | Use Case |
|-------|-------|-----------|----------|
| SeqCNN | (B, L, 4) one-hot | Conv1d | DNA classification |
| ProteinTransformer | (B, L) token ids | TransformerEncoder | Protein function |
| ESM2 (650M) | (B, L) residue ids | 33 Transformer layers | Transfer learning |
| LoRA | Any Linear layer | Low-rank A,B matrices | Parameter-efficient FT |

## Fine-Tuning Strategy Decision Tree
```
Is the dataset large (>100K sequences)?
├── Yes → Full fine-tuning with small LR (1e-4)
└── No → LoRA or freeze-head approach
    ├── Task similar to pretraining → Freeze more layers
    └── Task different from pretraining → Unfreeze more layers
```

---

## Interview Tip Bank
> **LoRA parameter count**: For Linear(d_in, d_out) with LoRA rank r: original = d_in×d_out params, LoRA adds r×(d_in+d_out) params. For d=768, r=8: original=589,824, LoRA adds 12,288 (~2%).

> **Why CLS token**: Instead of pooling all token embeddings, the CLS token attends to ALL other tokens during all Transformer layers, aggregating global information. Cleaner for classification.

> **Catastrophic forgetting fix**: (1) Small LR, (2) Gradual unfreezing, (3) Elastic weight consolidation (EWC), (4) Replay buffer. Most practical: small LR + gradual unfreeze.

> **Batch norm vs Layer norm**: BatchNorm depends on batch statistics — bad for NLP/protein tasks with variable-length sequences. LayerNorm normalizes per sample — works for any sequence length.

---

## Self-Check
1. A Conv1d(4, 32, kernel_size=7, padding=3) on input (B, 4, 200) — what's the output shape?
2. Why do we call `optimizer.zero_grad()` at the START of each batch, not after loss.backward()?
3. LoRA with rank r=4 on a Linear(768, 768): how many trainable parameters does LoRA add?
4. What is the key difference between Pre-LN and Post-LN in Transformers?
5. In ESM2 fine-tuning, why is 1e-5 a better LR than 1e-3?


## ⚡ Quick Jargon Guide

Terms used in this notebook that you may not know yet:

- **tensor** — multi-dimensional array (like numpy array but with GPU support)
- **attention** — mechanism letting each element focus on relevant parts of input
- **convolution** — sliding filter that detects local patterns (edges, motifs)
- **pooling** — reducing spatial dimensions by taking max/mean of each region
- **residue** — one amino acid in a protein chain — the repeating unit
- **MSA** — Multiple Sequence Alignment — aligned homologous protein sequences
- **invariant** — output unchanged when input is rotated — for scalar predictions
- **prior** — probability distribution before seeing any data

*Full definitions in the Concepts for Beginners table below (if present) or in the referenced prerequisite notebooks.*

Training a deep learning model from scratch on biological data can take weeks and millions of labelled examples. Fine-tuning takes a model that already understands biology and adapts it to your specific problem in hours. This notebook teaches both approaches.


## TL;DR — Plain English

**What this notebook does:** Builds deep learning models for biological sequences from the ground up, then teaches parameter-efficient fine-tuning (LoRA) — the technique used to adapt large pre-trained models cheaply.

**After this notebook you can:**
- Build a 1D convolutional neural network to classify DNA sequences
- Implement LoRA (Low-Rank Adaptation) from scratch in PyTorch
- Use EarlyStopping and learning rate scheduling to train models robustly
- Visualise gradient flow to diagnose vanishing/exploding gradient problems

**Why this matters:**
- HackerRank: Deep learning implementation questions (CNN architecture, regularisation, training loops) are tested in ML Engineer assessments
- computational biology ML teams: LoRA fine-tuning of protein language models (ESM, OpenFold) is exactly how teams adapt foundation models to specific tasks — this is the core skill for Module 10

**Time:** ~4 hours | **Difficulty:** ⭐⭐⭐ | **Prerequisites:** 00/02 (ML fundamentals), basic PyTorch helpful

## 🧠 ML Concepts for Beginners

| Term | Plain English |
|------|--------------|
| **nn.Module** | Blueprint for any neural network in PyTorch — like a class template you inherit from |
| **forward()** | Defines what happens when data flows through the network (called automatically when you do `model(x)`) |
| **loss** | A number measuring how wrong the model is — training tries to minimize it |
| **optimizer** | Algorithm that adjusts weights to reduce loss (`Adam` = most popular default) |
| **batch** | A small group of examples processed together (e.g. 32 sequences at once) |
| **epoch** | One complete pass through the entire training dataset |
| **LoRA** | Fine-tuning trick: add small trainable low-rank matrices to a frozen large model — changes <1% of parameters |
| **embedding** | Converts a discrete token (a letter like 'A') into a dense vector of numbers the network can process |
| **gradient** | Direction and magnitude to adjust each weight in order to reduce loss |
| **backprop** | Algorithm computing gradients by working backwards through the network (chain rule) |
| **overfitting** | Model memorizes training data but fails on new examples — needs regularization |
| **dropout** | During training, randomly zeros out neurons — forces the network to learn redundant features |
| **EarlyStopping** | Stop training when validation loss stops improving — prevents wasted compute and overfitting |
| **learning rate** | How big a step the optimizer takes each update — too high = unstable, too low = slow |

## Beginner Teaching Frame

**Who should fully work through this notebook:** students with core ML foundations who are ready for modern neural networks.

**How to study it on a first pass:**
- prioritize tensors, forward pass, loss, and optimization loop understanding
- understand one transformer block before worrying about large model stacks
- treat LoRA as an efficiency idea first and an implementation detail second

**Common traps:** memorizing architecture names without understanding data flow, and skipping over overfitting / optimization behavior.


## Canonical Resource Upgrade

Use these for reinforcement:
- [PyTorch Tutorials](https://docs.pytorch.org/tutorials/)
- [Dive into Deep Learning](https://d2l.ai/)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) after you understand the basics


## 📄 Primary Literature — Key Papers

These results are based on peer-reviewed publications. Read the originals to go deeper:

- **Vaswani et al. 2017** — *Attention Is All You Need*  
  [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)
- **Hu et al. 2021** — *LoRA: Low-Rank Adaptation of Large Language Models*  
  [https://arxiv.org/abs/2106.09685](https://arxiv.org/abs/2106.09685)


## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | Module 00/06 — PyTorch fundamentals (tensors, autograd, nn.Module, DataLoader) |
| 📍 You Are Here | Module 05/01 — Deep Learning & Fine-Tuning (CNNs, LoRA, transformers, ESM2 fine-tuning) |
| ➡️ Next Steps | Module 06/01 — Structural ML & GNNs, Module 07/01 — AlphaFold3 core, Module 10/01 — OpenFold3 fine-tuning capstone |
| 🏁 Goal | Fine-tune protein language models with LoRA; diagnose training failures; implement production-grade DL patterns for bioinformatics |

### 🆕 From Scratch? Start Here:
1. [Andrej Karpathy — micrograd](https://github.com/karpathy/micrograd) (free, build autograd from scratch, ~4h)
2. [fast.ai Lesson 1](https://course.fast.ai/Lessons/lesson1.html) (free, practical PyTorch)
3. Module 00/06 — PyTorch fundamentals (this project)
4. Module 04/01 — ML for omics (classical ML baseline to beat)
5. **This notebook**

**Time:** 15–20h | **Difficulty:** ⭐⭐⭐⭐

### 🔗 Cross-References
- **Builds on:** 00/06 PyTorch nn.Module/DataLoader patterns, 04/01 train/val/test split discipline and evaluation metrics
- **Used by:** 06/01 GNNs extend the message-passing concept to the PyG framework introduced here; 07/01 AlphaFold3 architecture review assumes transformer literacy; 10/01 OpenFold fine-tuning is the direct application of LoRA patterns from this notebook
- **Parallel reading:** 03/01 structural biology (structures that these models predict)


## What This Notebook Teaches (Plain English)

**Deep learning** = ML with many stacked layers that can learn complex patterns (images, sequences, text) without hand-crafted features.

**Fine-tuning** = taking a model already trained on millions of examples and adapting it to your specific task with far less data.

### Why It Matters for Biology

| What | Example |
|------|---------|
| CNN on DNA sequences | Predict transcription factor binding sites from raw ACGT sequence |
| Pre-trained protein model | ESM-2: trained on 250M protein sequences → knows protein "grammar" |
| Fine-tuning ESM-2 | Add a small head → predict protein function with only 100 labeled examples |
| LoRA | Fine-tune a billion-parameter model on a laptop by updating only 0.1% of parameters |

### Quick Glossary

| Term | Plain English |
|------|--------------|
| **Neural network** | Layers of simple computations (matrix multiply + nonlinearity) stacked to learn complex patterns |
| **Layer** | One transformation step: takes input vector, outputs new vector |
| **Weight / parameter** | A number the network learns during training |
| **Backpropagation** | Algorithm to compute how to adjust each weight to reduce error |
| **Epoch** | One full pass through the training dataset |
| **Batch** | A small group of samples processed together (e.g. 32 at a time) |
| **CNN** | Convolutional Neural Network: scans for local patterns (e.g. DNA motifs) |
| **Transformer** | Architecture using "attention" to find long-range dependencies (powers GPT, ESM-2) |
| **Fine-tuning** | Training a pre-trained model on new data, usually with very low learning rate |
| **LoRA** | Low-Rank Adaptation: insert tiny trainable matrices into frozen model layers |
| **Transfer learning** | Reuse knowledge from one task (e.g. predicting amino acid identity) for another (e.g. function) |

### What You Need Before Starting
- ML Fundamentals (Notebook 00/02)
- NumPy + basic Python
- Having seen a neural network diagram before helps but is not required

# Deep Learning & Fine-Tuning for Bioinformatics
**HackerRank Prep — Theme 5 | Interview-Grade**

Covers: CNNs for sequences, RNNs/Transformers, transfer learning, LoRA fine-tuning, ESM protein language model fine-tuning concepts.

> **Interview tip:** Know *when* to use DL vs classical ML, and *why* fine-tuning beats training from scratch.

---

> **Let's pause and recap.** So far you've learned:
>
> 1. Quick Jargon Guide

Terms used in this notebook that you may not know yet:

- **tensor** — multi-dimensional array (like numpy array but with GPU support)
- **attention** — mechanism letting each element focus on relevant parts of input
- **convolution** — sliding filter that detects local patterns (edges, motifs)
- **pooling** — reducing spatial dimensions by taking max/mean of each region
- **residue** — one amino acid in a protein chain — the repeating unit
- **MSA** — Multiple Sequence Alignment — aligned homologous protein sequences
- **invariant** — output unchanged when input is rotated — for scalar predictions
- **prior** — probability distribution before seeing any data

*Full definitions in the Concepts for Beginners table below (if present) or in the referenced prerequisite notebooks.*
> 2. ML Concepts for Beginners
> 3. Beginner Teaching Frame
>
> If any of these feel unclear, scroll back and re-read before continuing — the second half builds directly on them.

## 1. CNN for DNA Sequence Classification

In [ ]:

# ── Section 1: CNN for DNA Sequence Classification ─────────────────────────
# Architecture: Embedding(4,16) → Conv1d(16,32,k=5) → ReLU → MaxPool →
#               Conv1d(32,64,k=3) → AdaptiveAvgPool → Linear(64,2)
# Biology: DNA has 4 bases (A, C, G, T). We classify sequences as
#          promoter-active (GC-rich) vs inactive.

import torch                    # PyTorch: the main deep learning library
import torch.nn as nn           # nn: neural network building blocks (layers, losses)
import torch.nn.functional as F # F: stateless functions (relu, softmax, cross_entropy)

# --- CNN1D class definition ---
class CNN1D(nn.Module):
    """1-D convolutional classifier for DNA sequences."""
    # --- __init__() ---
    def __init__(self, vocab_size: int = 4, embed_dim: int = 16,
                 n_classes: int = 2, seq_len: int = 50):
        """
        Args:
            vocab_size: number of distinct tokens (4 for DNA: A/C/G/T)
            embed_dim:  embedding dimension (each token → a vector of this size)
            n_classes:  number of output classes (2 for binary classification)
            seq_len:    input sequence length (used to size the model)
        """
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, embed_dim)           # (B,L) → (B,L,16)
        self.conv1   = nn.Conv1d(embed_dim, 32, kernel_size=5, padding=2)  # (B,16,L)→(B,32,L)
        self.pool1   = nn.MaxPool1d(2)                                # (B,32,L/2)
        self.conv2   = nn.Conv1d(32, 64, kernel_size=3, padding=1)   # (B,32,L/2)→(B,64,L/2)
        self.gap     = nn.AdaptiveAvgPool1d(1)                        # global avg pool → (B,64,1)
        self.fc      = nn.Linear(64, n_classes)

    # --- forward() ---
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:  x (B, L) integer token indices
        Returns: (B, n_classes) raw logits — pass through softmax for probabilities
        """
        x = self.embed(x)          # (B, L, 16)
        x = x.permute(0, 2, 1)    # (B, 16, L)  — Conv1d expects (B, C, L)
        x = F.relu(self.conv1(x)) # (B, 32, L)
        x = self.pool1(x)          # (B, 32, L/2)
        x = F.relu(self.conv2(x)) # (B, 64, L/2)
        x = self.gap(x).squeeze(-1)  # (B, 64)
        # Return result
        return self.fc(x)          # (B, 2)


model = CNN1D(seq_len=50)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("CNN1D architecture:")
print(model)
print(f"\nTotal parameters   : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

# Quick sanity-check forward pass
dummy_seq = torch.randint(0, 4, (2, 50))  # batch=2, seq_len=50
out = model(dummy_seq)
print(f"\nInput shape  : {dummy_seq.shape}")
print(f"Output shape : {out.shape}  (batch=2, classes=2)")


> **Why this code:** This function processes biological data, converting raw sequences into a format our analysis can use.

> **Live demo — follow along!** Run each line of the next cell one at a time (or mentally trace it). Don't just hit "Run All" — the learning happens when you watch each step execute.

In [ ]:

# ── CNN Training Loop: GC-content classification ────────────────────────────
# Label: 1 if GC-content > 50% (G+C bases dominate), else 0
# Biologically, GC-rich promoters are often more stable and common in
# thermophilic organisms.

import torch                    # PyTorch: the main deep learning library
# --- Imports ---
import torch.nn as nn           # nn: neural network building blocks (layers, losses)
# Apply softmax normalization
import torch.nn.functional as F # F: stateless functions (relu, softmax, cross_entropy)
import numpy as np              # numpy: numerical arrays and math operations
import matplotlib               # matplotlib: foundational plotting library
matplotlib.use('Agg')           # Agg: non-interactive backend (renders to file, safe in scripts)
import matplotlib.pyplot as plt # pyplot: MATLAB-style API for creating figures and plots

torch.manual_seed(42)

# ── Data generation ─────────────────────────────────────────────────────────
SEQ_LEN   = 50
N_TRAIN   = 800
N_VAL     = 200
BASE2IDX  = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

# --- gc_content() function ---
def gc_content(seq_indices):
    """
    Plain English: calculates the fraction of G and C bases in each sequence.

    Args:
        seq_indices: integer tensor of shape (N, L) where 1=C and 2=G
    Returns:
        float tensor of shape (N,) — GC fraction per sequence (0.0 to 1.0)
    """
    return ((seq_indices == 1) | (seq_indices == 2)).float().mean(dim=-1)

# --- make_dataset() function ---
def make_dataset(n, seq_len=SEQ_LEN):
    """
    Plain English: generates random DNA sequences and labels them by GC content.

    Args:
        n:       number of sequences to generate
        seq_len: length of each sequence (default SEQ_LEN)
    Returns:
        tuple (seqs, labels) — seqs shape (n, seq_len), labels shape (n,) of 0 or 1
    """
    seqs   = torch.randint(0, 4, (n, seq_len))
    labels = (gc_content(seqs) > 0.5).long()
    return seqs, labels

X_train, y_train = make_dataset(N_TRAIN)
X_val,   y_val   = make_dataset(N_VAL)

print(f"Train: {X_train.shape}, GC>50%: {y_train.float().mean():.2%}")
print(f"Val  : {X_val.shape},   GC>50%: {y_val.float().mean():.2%}")

# ── Re-use CNN1D from previous cell ─────────────────────────────────────────
model     = CNN1D(seq_len=SEQ_LEN)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS     = 5
BATCH_SIZE = 64
train_losses, val_losses = [], []

# --- Loop over range(EPOCHS) ---
for epoch in range(EPOCHS):
    # ── Training ──
    model.train()
    perm   = torch.randperm(N_TRAIN)
    ep_loss = 0.0
    # --- Loop over range(0, N_TRAIN, BATCH_SIZE) ---
    for i in range(0, N_TRAIN, BATCH_SIZE):
        idx   = perm[i:i+BATCH_SIZE]
        xb, yb = X_train[idx], y_train[idx]
        # Optimizer update
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        # Backpropagate gradients
        loss.backward()
        # Optimizer update
        optimizer.step()
        ep_loss += loss.item() * len(yb)
    train_losses.append(ep_loss / N_TRAIN)

    # ── Validation ──
    model.eval()
    with torch.no_grad():
        vl = criterion(model(X_val), y_val).item()
        va = (model(X_val).argmax(1) == y_val).float().mean().item()
    val_losses.append(vl)
    print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_losses[-1]:.4f}  "
          f"val_loss={vl:.4f}  val_acc={va:.2%}")

# ── Loss curve plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
# Visualization
ax.plot(range(1, EPOCHS+1), train_losses, 'o-', label='Train loss')
# Visualization
ax.plot(range(1, EPOCHS+1), val_losses,   's--', label='Val loss')
# Visualization
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
# Visualization
ax.set_title('CNN1D — DNA GC-content classification'); ax.legend()
# Visualization
plt.tight_layout()
# Visualization
plt.savefig('/tmp/cnn1d_loss.png', dpi=80)
# Visualization
plt.show()
print("Loss curve saved to /tmp/cnn1d_loss.png")



> 📋 **Expected output:** Training loss should **decrease** over epochs (e.g. from ~0.7 to ~0.1). Validation loss should also decrease but may plateau or increase slightly at the end (overfitting).
> ✅ Loss decreasing = model is learning. ❌ Loss stuck or NaN = check learning rate (try 1e-4).

## 🔧 Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `CUDA out of memory` during fine-tuning | Full model gradients exceed GPU RAM | Use LoRA (trains ~1% of parameters), reduce batch size, or enable gradient checkpointing |
| `RuntimeError: shape mismatch` in classification head | Pre-trained head has wrong output size | Replace the final layer: `model.classifier = nn.Linear(d_model, num_classes)` |
| Loss is `NaN` after a few epochs | Learning rate too high for fine-tuning | Use 1e-5 to 5e-5 for fine-tuning (10-100x smaller than training from scratch) |
| Validation loss increases while train loss decreases | Overfitting — too many epochs or too little data | Use EarlyStopping with patience=3-5, or increase dropout/regularization |
| `RuntimeError: expected Float but got Half` | Mixed precision dtype conflict | Wrap forward pass in `torch.cuda.amp.autocast()`, keep loss in float32 |
| Model predicts same class for all inputs | Class imbalance or learning rate too small | Check class distribution, use weighted loss, verify gradients are non-zero |

## 📚 Further Reading

- **Vaswani et al. (2017)** "Attention Is All You Need" — the Transformer paper. DOI: 10.48550/arXiv.1706.03762
- **Hu et al. (2021)** "LoRA: Low-Rank Adaptation of Large Language Models" — DOI: 10.48550/arXiv.2106.09685
- **He et al. (2016)** "Deep Residual Learning" — skip connections. DOI: 10.1109/CVPR.2016.90
- **fast.ai Practical Deep Learning** — [course.fast.ai](https://course.fast.ai) — free, best practical DL course
- **Stanford CS231n** — Convolutional Neural Networks for Visual Recognition — free lecture videos on YouTube
- **d2l.ai** — "Dive into Deep Learning" free interactive textbook with PyTorch code

## 🎤 Interview Questions

**Q1 (Easy):** What is the difference between training from scratch and fine-tuning a pre-trained model?
<details><summary>Answer</summary>
Training from scratch initialises weights randomly and learns everything from your dataset — requires lots of data and compute. Fine-tuning starts from a model pre-trained on a large dataset (e.g., ImageNet, UniRef), then continues training on your smaller dataset. The pre-trained weights already capture general features (edges, textures, amino acid properties), so fine-tuning needs less data and converges faster. You typically use a smaller learning rate to avoid destroying learned features.
</details>

**Q2 (Easy):** What is EarlyStopping and why is it essential for deep learning?
<details><summary>Answer</summary>
EarlyStopping monitors validation loss during training and stops when it hasn't improved for `patience` epochs. It prevents overfitting: without it, the model would keep memorising training data while validation performance degrades. Implementation: track best validation loss, count epochs without improvement, restore best weights when stopping. It's one of the most important regularization techniques in practice.
</details>

**Q3 (Medium):** Explain LoRA (Low-Rank Adaptation) and why it is memory-efficient for fine-tuning large models.
<details><summary>Answer</summary>
LoRA freezes all pre-trained weights and injects trainable low-rank matrices into attention layers. Instead of updating a weight matrix W (d x d), it learns two small matrices A (d x r) and B (r x d) where r << d (typically 4-64). The effective weight becomes W + BA. This reduces trainable parameters from d² to 2dr — e.g., for d=1024, r=8: from 1M to 16K parameters per layer. Memory savings come from: (1) frozen weights don't need optimizer states, (2) gradients only for tiny LoRA matrices, (3) LoRA matrices fit in GPU memory even when the full model doesn't.
</details>

**Q4 (Medium):** When fine-tuning a CNN for protein structure classification, which layers would you freeze and which would you train? Why?
<details><summary>Answer</summary>
Freeze early layers (conv1-3) which capture low-level features (edges, textures) that transfer well across domains. Train later layers (conv4+, fully connected) which capture task-specific high-level features. Always replace and train the final classification head. Strategy depends on dataset size: small dataset → freeze more layers; large dataset → fine-tune all layers with a discriminative learning rate (lower LR for early layers, higher for later). For protein contact maps, early layers' edge detectors transfer well from natural images.
</details>

**Q5 (Hard):** You're fine-tuning a protein language model (ESM-2, 650M parameters) on 500 labelled sequences for secondary structure prediction. Design the fine-tuning strategy, addressing potential issues.
<details><summary>Answer</summary>
Strategy: (1) Use LoRA with rank 8-16 on attention layers — 500 samples can't support full fine-tuning of 650M params. (2) Add a lightweight classification head (Linear → ReLU → LayerNorm → Linear, per-residue output). (3) Freeze embeddings and first 50% of transformer layers. (4) Learning rate: 5e-5 for LoRA, 1e-3 for the new head (discriminative LR). (5) Use EarlyStopping on per-residue validation accuracy with patience=5. (6) Data augmentation: crop sequences to random windows, use homologous sequences from UniRef clusters. Issues: (a) 500 sequences may overfit even with LoRA — monitor train/val gap closely. (b) Class imbalance (helix/sheet/coil) — use weighted crossentropy. (c) Long sequences may not fit in memory — use gradient accumulation. Expected: Q3 accuracy ~80%+ leveraging ESM-2's pre-trained representations.
</details>

## ✅ Notebook Complete

**You can now:**
- Fine-tune pre-trained models with frozen layers and discriminative learning rates
- Implement LoRA for parameter-efficient fine-tuning
- Use EarlyStopping, learning rate scheduling, and gradient clipping
- Choose between CNNs, LSTMs, and Transformers for biological sequence tasks

**Next:** → `05_deep_learning_finetuning/02_sequence_models_rnn_lstm.ipynb`